In [ ]:
source ~/.bashrc

In [38]:
#-----------------------------------------------
# References:
# mae : https://github.com/facebookresearch/mae.git
# timm: https://github.com/rwightman/pytorch-image-models/tree/master/timm
#-----------------------------------------------

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

class simDecoder(nn.Module):

    def __init__(
        self,
        decoder_embed_dim: int = 768,
        img_size: int = 224,
        patch_size: int = 16,
        in_channels: int = 3,
    ):
        super().__init__()
        
        self.patch_size = patch_size
        self.img_size = img_size
        self.num_patches_per_dim = img_size // patch_size  

        self.decoder = nn.Sequential(nn.Conv2d(
            in_channels=decoder_embed_dim,
            out_channels=patch_size ** 2 * 3, 
            kernel_size=1),
            nn.PixelShuffle(patch_size))
            

    def forward(self, x, ids_restore, mask_token):
        """
        Args:
            x: (B, N_visible, D) - visible patch embeddings
            ids_restore: (B, N) - indices to restore full sequence
            mask_token: (1, 1, D) - learnable mask token
        Returns:
            pred: (B, C, H, W) - reconstructed image
        """
        B, N_visible, D = x.shape
        N = self.num_patches_per_dim ** 2
        
        # Expand mask token for all masked positions
        mask_tokens = mask_token.repeat(B, N - N_visible, 1)
        
        # Concatenate visible and mask tokens
        x_full = torch.cat([x, mask_tokens], dim=1)  # (B, N, D)
        
        # Restore original order using ids_restore
        x_full = torch.gather(
            x_full, 
            dim=1, 
            index=ids_restore.unsqueeze(-1).repeat(1, 1, D)
        )  # (B, N, D)
        
        # Reshape to spatial grid: (B, D, H, W)
        H = W = self.num_patches_per_dim
        x_spatial = x_full.transpose(1, 2).reshape(B, D, H, W)

        pred = self.decoder(x_spatial)

        return pred


class CF_MAE(nn.Module):
    """
    Temporal MAE with CONVOLUTIONAL decoder 
    """
    
    def __init__(
        self,
        img_size: int = 224,
        patch_size: int = 16,
        in_channels: int = 3,
        encoder_name: str = 'vit_base_patch16_224',
        encoder_embed_dim: int = 768,
        decoder_embed_dim: int = 768,
        mask_ratio: float = 0.75,
    ):
        super().__init__()
        
        self.num_patches = (img_size // patch_size) ** 2
        self.encoder_embed_dim = encoder_embed_dim
        self.patch_size = patch_size
        self.mask_ratio = mask_ratio
        
        # Shared encoder
        self.encoder = timm.create_model(
            encoder_name,
            pretrained=False,
            num_classes=0,
            global_pool='',
        )
        
        # Projections for latent alignment
        self.pre_projection = nn.Sequential(
            nn.Linear(encoder_embed_dim, encoder_embed_dim),
            nn.GELU(),
            nn.Linear(encoder_embed_dim, encoder_embed_dim),
        )
        
        self.post_projection = nn.Sequential(
            nn.Linear(encoder_embed_dim, encoder_embed_dim),
            nn.GELU(),
            nn.Linear(encoder_embed_dim, encoder_embed_dim),
        )
        
        # Change encoder (rich difference)
        self.change_encoder = nn.Sequential(
            nn.Linear(encoder_embed_dim * 4, encoder_embed_dim * 2),
            nn.GELU(),
            nn.Linear(encoder_embed_dim * 2, encoder_embed_dim),
        )
        
        # Decoder embedding (projects concatenated features)
        self.decoder_embed = nn.Linear(
            encoder_embed_dim * 2,  # z_pre + z_diff
            decoder_embed_dim
        )
        
        # Mask token
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        
        # CONVOLUTIONAL DECODER
        self.decoder = simDecoder(
            decoder_embed_dim=decoder_embed_dim,
            img_size=img_size,
            patch_size=patch_size,
            in_channels=in_channels,
        )
        
        self.initialize_weights()
    
    def initialize_weights(self):
        # Initialize mask token
        nn.init.normal_(self.mask_token, std=0.02)
        
        # Initialize linear layers
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)
    
    @torch.jit.ignore
    def no_weight_decay(self):
        return {'pos_embed', 'cls_token'}
    
    def random_masking(self, x, mask_ratio):
        """
        Perform random masking by per-sample shuffling.
        """
        B, N, D = x.shape
        len_keep = int(N * (1 - mask_ratio))
        
        noise = torch.rand(B, N, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        
        # Keep first subset
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))
        
        # Generate binary mask: 0 is keep, 1 is remove
        mask = torch.ones([B, N], device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)
        
        return x_masked, mask, ids_restore
    
    def encode(self, img, mask_ratio):
        """Encode image with masking"""
        # Get patch embeddings from encoder
        x = self.encoder.patch_embed(img)
        
        if hasattr(self.encoder, 'pos_embed'):
            pos = self.encoder.pos_embed[:, 1:, :]
            print("Used positional embedding:", pos.shape)
            x = x + pos


        # Random masking
        x, mask, ids_restore = self.random_masking(x, mask_ratio)
        
        # Apply encoder blocks
        for block in self.encoder.blocks:
            x = block(x)
        
        x = self.encoder.norm(x)
        
        return x, mask, ids_restore
    
    def compute_change_encoding(self, z_pre, z_post):
        """Compute rich difference encoding"""
        # Concatenate multiple change representations
        z_concat = torch.cat([
            z_pre,
            z_post,
            z_post - z_pre,  # Difference
            z_post * z_pre,  # Interaction
        ], dim=-1)
        
        z_diff = self.change_encoder(z_concat)
        return z_diff
    
    def forward(self, img_pre, img_post):
        # Encode both images
        z_pre_enc, mask_pre, ids_restore_pre = self.encode(img_pre, self.mask_ratio)
        z_post_enc, mask_post, ids_restore_post = self.encode(img_post, self.mask_ratio)
        
        # Project to aligned space
        z_pre = self.pre_projection(z_pre_enc)
        z_post = self.post_projection(z_post_enc)
        
        # Compute change encoding
        z_diff = self.compute_change_encoding(z_pre, z_post)
        
        # Concatenate z_pre and z_diff for decoder
        z_combined = torch.cat([z_pre, z_diff], dim=-1)
        z_dec = self.decoder_embed(z_combined)
        
        # CONVOLUTIONAL DECODE (returns full image, not patches)
        pred_post = self.decoder(z_dec, ids_restore_post, self.mask_token)
        print(f'[PRED SHAPE]------{pred_post.shape}')
        
        #Computing loss
        loss = self.compute_loss(img_post,pred_post,mask_post)
        
        return loss
    
    def compute_loss(
        self,
        img_post: torch.Tensor,
        pred_post: torch.Tensor,
        mask_post: torch.Tensor,
    ):
        """
        Compute loss for convolutional decoder
        Uses pixel-wise MSE instead of patch-wise
        """
        B, C, H, W = img_post.shape
        
        # Reshape mask to spatial dimensions
        mask_spatial = mask_post.reshape(B, H // self.patch_size, W // self.patch_size)
        mask_spatial = mask_spatial.unsqueeze(1).repeat(1, C, 1, 1)  # (B, C, h, w)
        
        # Upsample mask to full resolution
        mask_full = F.interpolate(
            mask_spatial.float(), 
            size=(H, W), 
            mode='nearest'
        )
        
        # Pixel-wise MSE loss on masked regions
        loss_recon = (pred_post - img_post) ** 2
        loss_recon = (loss_recon * mask_full).sum() / (mask_full.sum() + 1e-5)
        
        loss_dict = {'loss_recon': loss_recon}
    
        return loss_dict

In [39]:
model = CF_MAE()

In [36]:
x = torch.randint(0,255,(10,3,224,224), dtype=torch.float32)

In [40]:
model(x,x)

Used positional embedding: torch.Size([1, 196, 768])
Used positional embedding: torch.Size([1, 196, 768])
[PRED SHAPE]------torch.Size([10, 3, 224, 224])


{'loss_recon': tensor(21559.2988, grad_fn=<DivBackward0>)}

In [2]:
import torch
import torch.nn

In [73]:
class CF_MAE_SAMPLE(nn.Module):

    def __init__(self,img_size: int = 224,
        patch_size: int = 16,
        in_channels: int = 3,
        encoder_name: str = 'vit_base_patch16_224',
        encoder_embed_dim: int = 768,
        decoder_embed_dim: int = 768,
        mask_ratio: float = 0.75,):
        super().__init__()

        self.encoder_embed_dim = encoder_embed_dim
        self.num_patches = img_size // patch_size

        self.encoder = timm.create_model(model_name=encoder_name,pretrained=False,global_pool="")

        self.pre_projection = nn.Sequential(
            nn.Linear(encoder_embed_dim,encoder_embed_dim),
            nn.GELU(),
            nn.Linear(encoder_embed_dim,encoder_embed_dim)
        )
        self.post_projection = nn.Sequential(
            nn.Linear(encoder_embed_dim,encoder_embed_dim),
            nn.GELU(),
            nn.Linear(encoder_embed_dim,encoder_embed_dim)
        )
        self.change_encode = nn.Sequential(
            nn.Linear(encoder_embed_dim*4,encoder_embed_dim*2),
            nn.GELU(),
            nn.Linear(encoder_embed_dim*2,encoder_embed_dim)
        )
        self.decoder_proj = nn.Linear(encoder_embed_dim*2,encoder_embed_dim)

        self.mask_token = nn.Parameter(torch.zeros(1,1,encoder_embed_dim))

        self.simdecoder = nn.Sequential(nn.Conv2d(
            in_channels=encoder_embed_dim,
            out_channels=patch_size ** 2 * 3, 
            kernel_size=1),
            nn.PixelShuffle(patch_size))

    def change_encoding(self,pre,post):
        all_feature = torch.cat([pre,post,post-pre,post*pre],dim=-1)
        change = self.change_encode(all_feature)
        return change
    
    def random_masking(self, x, mask_ratio):
        """
        Perform random masking by per-sample shuffling.
        """
        B, N, D = x.shape
        len_keep = int(N * (1 - mask_ratio))
        
        noise = torch.rand(B, N, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        
        # Keep first subset
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))
        
        # Generate binary mask: 0 is keep, 1 is remove
        mask = torch.ones([B, N], device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)
        
        return x_masked, mask, ids_restore
    
    def encode(self,img,mask_ratio=0.75):
        x = self.encoder.patch_embed(img)
        
        if hasattr(self.encoder,"pos_embed"):
            x = x + self.encoder.pos_embed[:,1:,:]

        x, mask, ids_restore = self.random_masking(x,mask_ratio)

        for block in self.encoder.blocks:
            x = block(x)

        norm_x = self.encoder.norm(x)

        return x, mask, ids_restore
    
    def forward(self,pre,post):

        pre, mask_pre, ids_restore_pre = self.encode(pre)
        post, mask_pre, ids_restore_post =  self.encode(post)

        pre_proj = self.pre_projection(pre)
        post_proj = self.post_projection(post)

        zdiff = self.change_encoding(pre_proj,post_proj)

        x = torch.cat([pre_proj,zdiff],dim=-1)
        x = self.decoder_proj(x)
        B,N_visible,D = x.shape
        N = self.num_patches**2
        mask_tokens = self.mask_token.repeat(B,N-N_visible,1)
        x_full = torch.cat([x+mask_tokens],dim=1)
        x_full = torch.gather(x_full,dim=1,index= ids_restore_post.unsqueeze(-1).repeat(1, 1, D))
        H = W = self.num_patches
        x = x_full.transpose(1,2).reshape_as(B,D,H,W)

        x = self.simdecoder(x)
        print(x.shape)
        return x

In [74]:
model = CF_MAE_SAMPLE()

In [1]:
from model.CF_MAE import CF_MAE

c:\Users\HP\anaconda3\envs\AI4EO\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = CF_MAE()

In [3]:
model

CF_MAE(
  (encoder): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (norm): Identity()
          (proj): Linear(in_features=768, out_features=768, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (act):

In [4]:
import timm

In [ ]:
model = timm.create_model(model_name="vit_base_patch16_224",pretrained=True,drop_rate=0.1,drop_path_rate=0.1,global_pool="",)

In [6]:
model

VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (patch_drop): Identity()
  (norm_pre): Identity()
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (q_norm): Identity()
        (k_norm): Identity()
        (attn_drop): Dropout(p=0.0, inplace=False)
        (norm): Identity()
        (proj): Linear(in_features=768, out_features=768, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): Identity()
      (drop_path1): Identity()
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=768, out_features=3072, bias=True)
        (act): GELU(approximate='none')
        (drop1): Dropout(p=0.0, inplace=False